In [1]:
import pandas as pd
raw_vitals_file_path = '/home/guido/Code/charite/baselines/data/processed/raw_vitals.parquet'
raw_labs_file_path = '/home/guido/Code/charite/baselines/data/processed/raw_labs.parquet'
raw_vitals = pd.read_parquet(raw_vitals_file_path)
raw_labs = pd.read_parquet(raw_labs_file_path)

In [2]:
raw_vitals

,stay_id,subject_id,hours_since_icu_in,feature_name,value
0,37081114,10000690,0.283333,heart_rate,79.0
1,37081114,10000690,0.283333,sbp,107.0
2,37081114,10000690,0.283333,dbp,63.0
3,37081114,10000690,0.283333,respiratory_rate,23.0
4,37081114,10000690,0.283333,spo2,100.0
...,...,...,...,...,...
206461,34921121,10116099,41.850000,respiratory_rate,17.0
206462,34921121,10116099,41.850000,spo2,95.0
206463,34921121,10116099,42.850000,heart_rate,50.0
206464,34921121,10116099,42.850000,respiratory_rate,18.0


In [4]:

raw_vitals.nunique()

stay_id                 963
subject_id              710
hours_since_icu_in    37506
feature_name              6
value                   340
dtype: int64

In [5]:
raw_labs

,stay_id,subject_id,hours_since_icu_in,feature_name,value
0,37081114,10000690,7.316667,hematocrit,28.5
1,37081114,10000690,7.316667,hemoglobin,9.5
2,37081114,10000690,7.316667,platelets,199.0
3,37081114,10000690,7.316667,wbc,7.5
4,37081114,10000690,7.316667,chloride,104.0
...,...,...,...,...,...
36121,34921121,10116099,47.766667,creatinine,2.9
36122,34921121,10116099,47.766667,glucose,152.0
36123,34921121,10116099,47.766667,potassium,4.8
36124,34921121,10116099,47.766667,sodium,131.0


In [6]:
raw_labs.nunique()

stay_id                956
subject_id             704
hours_since_icu_in    5808
feature_name            11
value                  966
dtype: int64

In [ ]:
import numpy as np

test = np.load('/home/guido/Code/charite/baselines/data/processed/test.npz')



AttributeError: 'NpzFile' object has no attribute 'head'

In [12]:
test.keys()

KeysView(NpzFile '/home/guido/Code/charite/baselines/data/processed/test.npz' with keys: data, orig_mask, stay_ids, subject_ids)

In [7]:
import timesfm
import inspect
# Questo stamperà esattamente cosa si aspetta la tua versione di TimesFm
print(f"Argomenti accettati: {inspect.signature(timesfm.TimesFm.__init__)}")

Argomenti accettati: (self, hparams: timesfm.timesfm_base.TimesFmHparams, checkpoint: timesfm.timesfm_base.TimesFmCheckpoint) -> None


In [11]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
import timesfm

# --- 1. DOWNLOAD DEI PESI ---
repo_id = "google/timesfm-2.5-200m-pytorch"
model_dir = os.path.expanduser("~/timesfm_25_weights")

print(f"🚀 Controllo pesi in {model_dir}...")
# snapshot_download gestisce automaticamente se i file sono già presenti o vanno scaricati
snapshot_download(repo_id=repo_id, local_dir=model_dir)

# --- 2. GESTIONE E CONVERSIONE FORMATI ---
ckpt_path = os.path.join(model_dir, "torch_model.ckpt")
safetensors_path = os.path.join(model_dir, "model.safetensors")

if not os.path.exists(ckpt_path):
    if os.path.exists(safetensors_path):
        print("🔄 Conversione model.safetensors -> torch_model.ckpt in corso...")
        sd_to_convert = load_file(safetensors_path)
        torch.save(sd_to_convert, ckpt_path)
        print("✅ Conversione completata.")
    else:
        raise FileNotFoundError("Impossibile trovare i file dei pesi nel repository.")

# --- 3. TRADUZIONE STATE_DICT (Mappatura nomi layer) ---
print("🧠 Traduzione dei nomi dei layer per la nuova versione della libreria...")
old_sd = torch.load(ckpt_path, map_location="cpu")
new_sd = {}

# Mappa per convertire i nomi "Google Legacy" in "Hugging Face/New Standard"
mapping = {
    "tokenizer": "input_ff_layer",
    "stacked_xf": "stacked_transformer.layers",
    "output_projection_point": "horizon_ff_layer",
    "output_projection_quantiles": "horizon_ff_layer_quantiles",
    "attn.qkv_proj": "self_attn.qkv_proj",
    "attn.out": "self_attn.o_proj",
    "ff0": "mlp.gate_proj",
    "ff1": "mlp.down_proj",
    "pre_attn_ln": "input_layernorm",
    "pre_ff_ln": "mlp.layer_norm",
}

for k, v in old_sd.items():
    new_k = k
    for old, new in mapping.items():
        new_k = new_k.replace(old, new)
    # Aggiustamento per gli strati hidden che ora richiedono '.0.'
    if "hidden_layer" in new_k and ".0." not in new_k:
        new_k = new_k.replace("hidden_layer", "hidden_layer.0")
    new_sd[new_k] = v

# --- 4. INIZIALIZZAZIONE MODELLO ---
hparams = timesfm.TimesFmHparams(
    context_len=1024,
    horizon_len=12,
    input_patch_len=32,
    output_patch_len=128,
    num_layers=20,
    model_dims=1280,
    backend="torch",
)

# Creiamo l'oggetto checkpoint per

🚀 Controllo pesi in /home/guido/timesfm_25_weights...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00,  4.66it/s]


🔄 Conversione model.safetensors -> torch_model.ckpt in corso...
✅ Conversione completata.
🧠 Traduzione dei nomi dei layer per la nuova versione della libreria...


/home/guido/.tmp/ipykernel_1630751/750252215.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  old_sd = torch.load(ckpt_path, map_location="cpu")
